# 05_strategy_builder.ipynb - Fases 5-9: RF → Feature Selection → Tree → Reglas

Pipeline completo:
1. Split temporal 70/30
2. Random Forest (feature importance)
3. Select top 3-5 features
4. Decision Tree (max_depth=3/4)
5. Extract rules from leaves
6. Filter: P(TP)>0.5, n_samples>=30

In [1]:
import sys
import importlib
import pandas as pd
import numpy as np
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import src.strategy_builder as sb_mod
importlib.reload(sb_mod)
from src.strategy_builder import StrategyBuilder, StrategyRule
from src.labeling import create_labeled_dataset
from src.features import add_all_features
from src.data import load_ohlc_from_yfinance

from config.settings import DATA, FEATURES, RISK

print("Imports listos - scikit-learn disponible?", 'sklearn' in sys.modules or True)
print("StrategyBuilder cargado desde:", sb_mod.__file__)

Imports listos - scikit-learn disponible? True
StrategyBuilder cargado desde: /home/paolo/Escritorio/projecto_machine_learning_quant/smallcap-quant-ml/src/strategy_builder.py


## Ejecutar pipeline completo (FIXME - requiere instalar scikit-learn)

In [3]:
import glob
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
LABELED_DIR = REPO_ROOT / "data" / "labeled"
OUTPUT_JSON = REPO_ROOT / "data" / "strategies_candidates.json"

labeled_files = sorted(LABELED_DIR.glob("labeled_*.csv"))
print(f"Archivos etiquetados encontrados: {len(labeled_files)}")

if not labeled_files:
    raise FileNotFoundError(f"No se encontraron archivos etiquetados en {LABELED_DIR}")

train_frames = []
test_frames = []
split_summary = []

for path in labeled_files:
    ticker = path.stem.replace("labeled_", "")
    df = pd.read_csv(path)

    if "Datetime" in df.columns:
        df["Datetime"] = pd.to_datetime(df["Datetime"], utc=True, errors="coerce")
        df["Datetime"] = df["Datetime"].dt.tz_convert(None)

    df = df.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

    if len(df) < 20:
        print(f"{ticker}: se omite por tener muy pocas filas ({len(df)})")
        continue

    split_idx = int(len(df) * 0.7)
    if split_idx <= 0 or split_idx >= len(df):
        print(f"{ticker}: split inválido ({len(df)} filas)")
        continue

    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()

    train_frames.append(train_df)
    test_frames.append(test_df)
    split_summary.append((ticker, len(train_df), len(test_df), split_idx))
    print(f"{ticker}: train={len(train_df)} | test={len(test_df)}")

if not train_frames:
    raise RuntimeError("No hay datasets válidos para entrenar")

train_df = pd.concat(train_frames, ignore_index=True)
test_df = pd.concat(test_frames, ignore_index=True)

train_df["tp_target"] = (train_df["label"] == 1).astype(int)
test_df["tp_target"] = (test_df["label"] == 1).astype(int)

print("\nDistribución del target binario:")
print(train_df["tp_target"].value_counts().sort_index())

print(f"\nTrain combinado: {len(train_df)} filas")
print(f"Test combinado: {len(test_df)} filas")

summary_df = pd.DataFrame(split_summary, columns=["ticker", "train_rows", "test_rows", "split_idx"])
summary_df

builder = StrategyBuilder(
    top_n_features=5,
    max_depth=4,
    min_samples_leaf=10,
    random_state=42,
    positive_label=1,
    min_positive_rate=0.20,
)
rules = builder.fit(train_df, label_col="tp_target")

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
builder.export_strategies(str(OUTPUT_JSON))
print(f"\nEstrategias guardadas en: {OUTPUT_JSON}")
print(f"Total de reglas válidas extraídas: {len(rules)}")

Archivos etiquetados encontrados: 9
ABSI: train=3522 | test=1510
EBON: train=2470 | test=1059
HBIO: train=1721 | test=738
HOTH: train=2183 | test=936
IMUX: train=3497 | test=1500
KPTI: train=3238 | test=1389
LRMR: train=3522 | test=1510
MCRB: train=2545 | test=1092
SGMO: train=583 | test=251

Distribución del target binario:
tp_target
0    15539
1     7742
Name: count, dtype: int64

Train combinado: 23281 filas
Test combinado: 9985 filas
Extracted 11 valid strategies from 23281 samples
  - Rule 0: ['VWAP_20 <= 2.162', 'ATR_50 <= 0.047', 'ATR_50 <= 0.035', 'VWAP_20 <= 1.355'] (n=604, P(TP)=0.209)
  - Rule 1: ['VWAP_20 <= 2.162', 'ATR_50 <= 0.047', 'ATR_50 > 0.035', 'EMA_20 <= 1.294'] (n=400, P(TP)=0.405)
  - Rule 2: ['VWAP_20 <= 2.162', 'ATR_50 <= 0.047', 'ATR_50 > 0.035', 'EMA_20 > 1.294'] (n=683, P(TP)=0.266)
  - Rule 3: ['VWAP_20 <= 2.162', 'ATR_50 > 0.047', 'VWAP_20 <= 1.362', 'SMA_10 <= 1.016'] (n=10, P(TP)=0.200)
  - Rule 4: ['VWAP_20 <= 2.162', 'ATR_50 > 0.047', 'VWAP_20 > 1.362'

## Salida esperada

Cada estrategia válida tendrá:
- conditions: ['RSI_14 > 55', 'MACD_line > 0', ...]
- n_samples: >= 30
- p_label_1: > 0.5
- rule_id: identificador único